|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Quantization<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: quantize it, then find the speed you lost<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
import cudalib

torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(dev)

Quantize the weights, then find out why it did not make anything faster.

Stages 18 and 18b. The first half is arithmetic and the second half is a
measurement that should annoy you.

# Exercise 1: INT8, per output channel

In [ ]:
def quantize_int8(W):
  scales = W.abs().amax(dim=1).clamp(min=1e-8) / 127.0
  q = torch.round(W / scales[:, None]).clamp(-127, 127).to(torch.int8)
  return q, scales.float()

def dequantize_int8(q, scales):
  return q.float() * scales[:, None]

W = torch.randn(512, 1024, device=dev)
q, s = quantize_int8(W)
assert q.dtype == torch.int8 and s.shape == (512,)
rel = ((dequantize_int8(q,s) - W).abs().mean()/W.abs().mean()).item()
print(f'mean relative error {rel:.4f}')
print(f'bytes: {W.numel()*4/1e6:.2f} MB fp32 -> {(q.numel()+s.numel()*4)/1e6:.2f} MB')

# Exercise 2: why per channel and not per tensor

In [ ]:
Wo = W.clone(); Wo[0] *= 100.0

st = Wo.abs().max()/127.0
err_tensor = ((torch.round(Wo/st).clamp(-127,127)*st - Wo).abs().mean()
              / Wo.abs().mean()).item()

qc, sc = quantize_int8(Wo)
err_chan = ((dequantize_int8(qc,sc) - Wo).abs().mean()/Wo.abs().mean()).item()

print(f'per tensor  {err_tensor:.4f}')
print(f'per channel {err_chan:.4f}   ({err_tensor/err_chan:.0f}x better)')

# Exercise 3: now time it

Half the bytes should be most of half the time. Check.

In [ ]:
if dev == 'cuda':
  Wb = torch.randn(4096, 4096, device=dev, dtype=torch.bfloat16)
  qb, sb = quantize_int8(Wb.float())
  sbb = sb.to(torch.bfloat16)
  x = torch.randn(1, 4096, device=dev, dtype=torch.bfloat16)

  bf16    = cudalib.bench_ms(lambda: x @ Wb.t(), best_of=3)
  unfused = cudalib.bench_ms(lambda: x @ (qb.to(torch.bfloat16)*sbb[:,None]).t(), best_of=3)

  print(f'bf16 matmul:             {bf16:7.3f} ms')
  print(f'dequantize then matmul:  {unfused:7.3f} ms  ({unfused/bf16:.1f}x SLOWER)')
  print(f'\nhalving the weights made it {unfused/bf16:.1f} times slower.')
  print('Everything after this point in stage 18b is about why.')

### Two results, and the second one is the stage

**Per-channel scaling is worth about fifty times the accuracy** of a
single tensor scale, on a matrix with one outlier row. Real weight
matrices have outlier rows. It costs one float per output channel,
which is nothing.

**And quantizing made the multiply slower.** Not slightly: several times
slower than not quantizing at all. You halved the stored bytes and then
rebuilt them, so the memory system moved the small weights AND a
full-size bf16 copy, and the arithmetic units did the same work as
before.

This is the most common way quantization is deployed wrong, and the
symptom is that the model got smaller and the server got slower, which
nobody expects and everybody blames on something else.

The fix is the epilogue. The scale is per output channel, so it comes
out of the sum: apply it once, to a value already in a register, after
the dot product. Stage 18b makes you write that kernel.

    ./vc guide 18b